# Assisted business document suggestions (ES/PT/EN)

Unpublished candidate. The exact image passed private real-time and certified batch checks; commercial publication is pending.

Give a reviewer two type suggestions for one already extracted, known-supported business document. A person must choose or reject them. No OCR, unknown/mixed detection, authenticity checks or automated routing. Validation uses generated examples; real customer-document accuracy is unmeasured.

Default **Run All is offline**. To deploy after publication, subscribe in AWS Marketplace, copy your subscribed **us-east-1** model-package ARN into `DOCUMENT_CLASSIFIER_MODEL_PACKAGE_ARN`, supply an existing `SAGEMAKER_EXECUTION_ROLE_ARN`, review IAM/costs/cleanup, then set `RUN_DEMO=True`. AWS infrastructure charges are separate from software/trial terms. This example creates one temporary CPU endpoint and deletes its own resources in `finally`. If the kernel is terminated, use the printed name to verify/remove that endpoint in your account; charges continue while it exists. No S3 bucket, IAM role or data capture is created.

The optional Batch Transform section below also defaults off. It uses the included fictional three-language examples and an existing S3 bucket you authorize in us-east-1. It writes only a unique demo prefix, retains its small input/output objects for inspection, and cleans up the task model after the job stops. Your execution role needs read/write access to that prefix. Neither example creates a bucket or IAM role.

Seller-account rehearsal: a certified seller-owned source has no buyer ProductId.
The notebook verifies its actual source association with this Limited/Public
product and Released offer through read-only Catalog calls. Ordinary subscribed
packages follow their Marketplace identity path. Seller rehearsal does not prove
an independent buyer's checkout or entitlement, and no subscription is purchased.


In [ ]:
import json
import os
import re
import uuid

import boto3

# Preserve references if an earlier cleanup needs attention.
if any(
    globals().get(key, False)
    for key in ("model_created", "config_created", "endpoint_created", "batch_model_created", "batch_job_started")
):
    raise RuntimeError("Clean up the previous demo before rerunning setup; its resource names are preserved.")

# Default Run All performs no AWS requests and creates no resources.
RUN_DEMO = False
REGION = "us-east-1"
# Exact image verified by private real-time and certified batch execution.
APPROVED_IMAGE_DIGEST = "sha256:58d7c9ad8b9f56f1ccb2f313cc9380327a7e45d91a5af4dd972b7ea34d9da89b"
PACKAGE_ARN = os.environ.get("DOCUMENT_CLASSIFIER_MODEL_PACKAGE_ARN", "").strip()
ROLE_ARN = os.environ.get("SAGEMAKER_EXECUTION_ROLE_ARN", "").strip()
DEMO_RUN_NAME = (
    os.environ.get("DOCUMENT_CLASSIFIER_DEMO_RUN_NAME", "").strip() or "ss-document-demo-" + uuid.uuid4().hex[:12]
)
SAMPLE_REQUEST = {
    "instances": [
        {
            "filename": "document-025.txt",
            "id": "final-en-bank_statement-01",
            "text": "Business Account Activity, Period Ending 31 October\n"
            "Fictional Stonebridge Ledger Bank reports account 4408 for "
            "Northglass Exhibit Works. Opening balance was $9,840.00. "
            "Credits were $3,500.00; debits were $1,120.00 for equipment "
            "rental and $340.00 in service fees. Closing balance is "
            "$11,880.00. The cycle covers 1 through 31 October 2026. "
            "Reconcile outstanding entries before the next monthly close.",
        }
    ]
}


MARKETPLACE_PRODUCT_ID = "prod-c7nqgnwq4v55q"
MARKETPLACE_OFFER_ID = "offer-5xz2zondvkpts"


def validate_package(package, session, caller_account):
    if package.get("ModelPackageStatus") != "Completed":
        raise ValueError("The subscribed package is not Completed.")
    containers = package.get("InferenceSpecification", {}).get("Containers", [])
    if len(containers) != 1:
        raise ValueError("Use a single-container Marketplace model-package ARN.")
    digest = containers[0].get("ImageDigest") or containers[0].get("Image", "").partition("@")[2]
    if digest != APPROVED_IMAGE_DIGEST:
        raise ValueError("The subscribed image differs from this release.")
    if containers[0].get("ProductId"):
        return "marketplace_package"
    # A seller-owned source package has no buyer ProductId. Verify its actual
    # association instead of pretending a subscription was purchased.
    arn = package.get("ModelPackageArn", "")
    if not (
        package.get("CertifyForMarketplace") is True
        and arn.startswith(f"arn:aws:sagemaker:{REGION}:{caller_account}:model-package/")
    ):
        raise ValueError("Use your subscribed Marketplace model-package ARN.")
    catalog = session.client("marketplace-catalog")
    product = json.loads(catalog.describe_entity(Catalog="AWSMarketplace", EntityId=MARKETPLACE_PRODUCT_ID)["Details"])
    offer = json.loads(catalog.describe_entity(Catalog="AWSMarketplace", EntityId=MARKETPLACE_OFFER_ID)["Details"])
    sources = [
        source
        for version in product.get("Versions", [])
        if version.get("VersionTitle") == "1.0.0"
        for source in version.get("Sources", [])
    ]
    if not (
        product.get("Description", {}).get("Visibility") in {"Limited", "Public"}
        and len(sources) == 1
        and sources[0].get("ModelPackageArn") == arn
        and offer.get("ProductId") == MARKETPLACE_PRODUCT_ID
        and offer.get("State") == "Released"
    ):
        raise ValueError("Seller package is not associated with the released Marketplace offer.")
    print("Verified seller-account Marketplace source; this is not a buyer subscription test.")
    return "seller_owned_marketplace_source"

In [ ]:
if not RUN_DEMO:
    print("Offline preview only. Review costs and set RUN_DEMO=True to create a temporary endpoint.")
else:
    if any(globals().get(key, False) for key in ("model_created", "config_created", "endpoint_created")):
        raise RuntimeError("Clean up the previous demo before creating another endpoint.")
    if REGION != "us-east-1" or not re.fullmatch(r"ss-document-demo-[0-9a-f]{12}", DEMO_RUN_NAME):
        raise ValueError("Use the qualified us-east-1 region and a unique demo run name.")
    if not re.fullmatch(r"sha256:[0-9a-f]{64}", APPROVED_IMAGE_DIGEST):
        raise ValueError("Unreleased candidate: no qualified image digest is configured.")
    if not re.fullmatch(r"arn:aws:sagemaker:us-east-1:\d{12}:model-package/[A-Za-z0-9-]+", PACKAGE_ARN):
        raise ValueError("Set DOCUMENT_CLASSIFIER_MODEL_PACKAGE_ARN to your subscribed us-east-1 ARN.")
    session = boto3.Session(region_name=REGION)
    account_id = session.client("sts").get_caller_identity()["Account"]
    if not ROLE_ARN.startswith(f"arn:aws:iam::{account_id}:role/"):
        raise ValueError("Provide an existing SageMaker execution role from your account.")
    sm = session.client("sagemaker")
    validate_package(sm.describe_model_package(ModelPackageName=PACKAGE_ARN), session, account_id)
    name = DEMO_RUN_NAME
    tags = [{"Key": "SoftwareSushiDemoRun", "Value": name}]
    model_created = config_created = endpoint_created = False
    print("Creating", name, "on one ml.c5.xlarge. Cleanup runs in finally; retain this name.")
    try:
        sm.create_model(
            ModelName=name,
            ExecutionRoleArn=ROLE_ARN,
            PrimaryContainer={"ModelPackageName": PACKAGE_ARN},
            EnableNetworkIsolation=True,
            Tags=tags,
        )
        model_created = True
        sm.create_endpoint_config(
            EndpointConfigName=name,
            ProductionVariants=[
                {
                    "VariantName": "AllTraffic",
                    "ModelName": name,
                    "InstanceType": "ml.c5.xlarge",
                    "InitialInstanceCount": 1,
                    "InitialVariantWeight": 1,
                    "ContainerStartupHealthCheckTimeoutInSeconds": 180,
                    "ModelDataDownloadTimeoutInSeconds": 300,
                }
            ],
            Tags=tags,
        )
        config_created = True
        sm.create_endpoint(EndpointName=name, EndpointConfigName=name, Tags=tags)
        endpoint_created = True
        sm.get_waiter("endpoint_in_service").wait(EndpointName=name, WaiterConfig={"Delay": 15, "MaxAttempts": 120})
        response = session.client("sagemaker-runtime").invoke_endpoint(
            EndpointName=name, ContentType="application/json", Body=json.dumps(SAMPLE_REQUEST).encode()
        )
        result = json.loads(response["Body"].read())
        print(json.dumps(result, indent=2, ensure_ascii=False))
        assert result["predictions"][0]["selection_required"] is True
        print("Review both suggestions and explicitly choose or reject them in your application.")
    finally:
        # Only the uniquely named resources created by this invocation are removed.
        if endpoint_created:
            current = sm.describe_endpoint(EndpointName=name)
            if current["EndpointConfigName"] != name:
                raise RuntimeError("Endpoint identity differs; preserve it for manual review.")
            sm.delete_endpoint(EndpointName=name)
            sm.get_waiter("endpoint_deleted").wait(EndpointName=name, WaiterConfig={"Delay": 15, "MaxAttempts": 120})
            endpoint_created = False
        if config_created:
            current = sm.describe_endpoint_config(EndpointConfigName=name)
            if len(current["ProductionVariants"]) != 1 or current["ProductionVariants"][0]["ModelName"] != name:
                raise RuntimeError("Endpoint configuration differs; preserve it for manual review.")
            sm.delete_endpoint_config(EndpointConfigName=name)
            config_created = False
        if model_created:
            current = sm.describe_model(ModelName=name)
            if current["PrimaryContainer"].get("ModelPackageName") != PACKAGE_ARN:
                raise RuntimeError("Model package identity differs; preserve it for manual review.")
            sm.delete_model(ModelName=name)
            model_created = False
        print("Temporary demo cleanup completed:", name)

## Optional Batch Transform example

Set `DOCUMENT_CLASSIFIER_BATCH_BUCKET` to an existing authorized bucket in
us-east-1, review storage/compute/software charges, then enable `RUN_BATCH_DEMO`.
The included ES/PT/EN text is fictional. No external dataset or file upload is
needed. Keep the printed job/model/prefix for inspection. Input/output S3 objects
are retained; remove those exact demo objects yourself when no longer needed.
The code uses line splitting, SingleRecord, one C5 instance and concurrency one.


In [ ]:
# Default Run All remains offline.
RUN_BATCH_DEMO = False
BATCH_BUCKET = os.environ.get("DOCUMENT_CLASSIFIER_BATCH_BUCKET", "").strip()
BATCH_REQUESTS = [
    {
        "instances": [
            {
                "filename": "document-025.txt",
                "id": "final-en-bank_statement-01",
                "text": (
                    "Business Account Activity, Period Ending 31 October\nFictional Sto"
                    "nebridge Ledger Bank reports account 4408 for Northglass Exhibit "
                    "Works. Opening balance was $9,840.00. Credits were $3,500.00; deb"
                    "its were $1,120.00 for equipment rental and $340.00 in service fe"
                    "es. Closing balance is $11,880.00. The cycle covers 1 through 31 "
                    "October 2026. Reconcile outstanding entries before the next month"
                    "ly close."
                ),
            }
        ]
    },
    {
        "instances": [
            {
                "filename": "document-025.txt",
                "id": "final-es-bank_statement-01",
                "text": (
                    "Estado de cuenta empresarial, período del 1 al 31 de enero. Coope"
                    "rativa Horizonte registra saldo inicial de MXN 84.200, depósitos "
                    "por 126.000 y débitos por 97.450. Se detallan nómina, renta de of"
                    "icina y dos abonos de clientes. Saldo final: MXN 112.750. No hay "
                    "cargos pendientes al cierre; las operaciones pueden objetarse dur"
                    "ante los treinta días siguientes."
                ),
            }
        ]
    },
    {
        "instances": [
            {
                "filename": "document-025.txt",
                "id": "final-pt-bank_statement-01",
                "text": (
                    "EXTRATO EMPRESARIAL — Cooperativa Horizonte de Barro, conta de mo"
                    "vimento 00418-7, período de 1 a 15 de novembro. Saldo inicial: R$"
                    " 12.460,00. Entradas identificadas: vendas balcão, R$ 8.320,00; r"
                    "epasse Feira do Vale, R$ 1.180,00. Saídas: aluguel do galpão, R$ "
                    "2.100,00; fornecedor Argila Serena, R$ 860,00. Saldo final dispon"
                    "ível: R$ 19.000,00."
                ),
            }
        ]
    },
]

if not RUN_BATCH_DEMO:
    print("Offline batch preview only. Supply an authorized S3 bucket and explicitly enable the demo.")
else:
    if any(globals().get(key, False) for key in ("batch_model_created", "batch_job_started")):
        raise RuntimeError("Previous batch cleanup is incomplete; resource references are preserved.")
    if REGION != "us-east-1" or not re.fullmatch(r"ss-document-demo-[0-9a-f]{12}", DEMO_RUN_NAME):
        raise ValueError("Use the qualified region and a unique task-owned demo name.")
    if not re.fullmatch(r"[a-z0-9][a-z0-9.-]{1,61}[a-z0-9]", BATCH_BUCKET):
        raise ValueError("Supply an existing authorized S3 batch bucket.")
    if not re.fullmatch(r"arn:aws:sagemaker:us-east-1:\d{12}:model-package/[A-Za-z0-9-]+", PACKAGE_ARN):
        raise ValueError("Set the subscribed Marketplace package ARN before opting in.")
    session = boto3.Session(region_name=REGION)
    account_id = session.client("sts").get_caller_identity()["Account"]
    if not ROLE_ARN.startswith(f"arn:aws:iam::{account_id}:role/"):
        raise ValueError("Provide an existing SageMaker execution role from your account.")
    sm, s3 = session.client("sagemaker"), session.client("s3")
    validate_package(sm.describe_model_package(ModelPackageName=PACKAGE_ARN), session, account_id)
    if (s3.get_bucket_location(Bucket=BATCH_BUCKET)["LocationConstraint"] or "us-east-1") != REGION:
        raise ValueError("Use an S3 bucket in the qualified region.")
    batch_name = DEMO_RUN_NAME + "-batch"
    batch_prefix = "document-classifier-qualification/notebook/" + DEMO_RUN_NAME
    input_key = batch_prefix + "/input/examples.jsonl"
    output_prefix = batch_prefix + "/output/"
    payload = "".join(json.dumps(row, ensure_ascii=False) + "\n" for row in BATCH_REQUESTS).encode()
    # Never overwrite an earlier run's input, even if a name is accidentally reused.
    s3.put_object(
        Bucket=BATCH_BUCKET,
        Key=input_key,
        Body=payload,
        ContentType="application/jsonlines",
        ServerSideEncryption="AES256",
        IfNoneMatch="*",
    )
    batch_model_created = batch_job_started = False
    print("Batch model/job:", batch_name, "S3 prefix:", batch_prefix)
    try:
        sm.create_model(
            ModelName=batch_name,
            ExecutionRoleArn=ROLE_ARN,
            PrimaryContainer={"ModelPackageName": PACKAGE_ARN},
            EnableNetworkIsolation=True,
            Tags=[{"Key": "SoftwareSushiDemoRun", "Value": DEMO_RUN_NAME}],
        )
        batch_model_created = True
        sm.create_transform_job(
            TransformJobName=batch_name,
            ModelName=batch_name,
            BatchStrategy="SingleRecord",
            MaxConcurrentTransforms=1,
            MaxPayloadInMB=1,
            ModelClientConfig={"InvocationsTimeoutInSeconds": 60, "InvocationsMaxRetries": 0},
            TransformInput={
                "DataSource": {"S3DataSource": {"S3DataType": "S3Prefix", "S3Uri": f"s3://{BATCH_BUCKET}/{input_key}"}},
                "ContentType": "application/jsonlines",
                "CompressionType": "None",
                "SplitType": "Line",
            },
            TransformOutput={
                "S3OutputPath": f"s3://{BATCH_BUCKET}/{output_prefix}",
                "Accept": "application/jsonlines",
                "AssembleWith": "Line",
            },
            TransformResources={"InstanceType": "ml.c5.xlarge", "InstanceCount": 1},
            Tags=[{"Key": "SoftwareSushiDemoRun", "Value": DEMO_RUN_NAME}],
        )
        batch_job_started = True
        sm.get_waiter("transform_job_completed_or_stopped").wait(
            TransformJobName=batch_name, WaiterConfig={"Delay": 15, "MaxAttempts": 120}
        )
        job = sm.describe_transform_job(TransformJobName=batch_name)
        if job["TransformJobStatus"] != "Completed":
            raise RuntimeError("Batch did not complete successfully; inspect the task job.")
        raw = s3.get_object(Bucket=BATCH_BUCKET, Key=output_prefix + "examples.jsonl.out")["Body"].read()
        results = [json.loads(line) for line in raw.splitlines()]
        assert len(results) == len(BATCH_REQUESTS)
        for request, response in zip(BATCH_REQUESTS, results, strict=True):
            assert len(response["predictions"]) == 1
            row = response["predictions"][0]
            assert row["id"] == request["instances"][0]["id"]
            assert row["selection_required"] is True and row["needs_review"] is True
            assert len(row["suggestions"]) == 2 and row["validation_basis"] == "generated_only"
        print(json.dumps(results, indent=2, ensure_ascii=False))
    finally:
        if batch_job_started:
            current = sm.describe_transform_job(TransformJobName=batch_name)
            if current["ModelName"] != batch_name:
                raise RuntimeError("Job model identity differs; preserve resources for review.")
            if current["TransformJobStatus"] == "InProgress":
                sm.stop_transform_job(TransformJobName=batch_name)
            if current["TransformJobStatus"] not in {"Completed", "Failed", "Stopped"}:
                sm.get_waiter("transform_job_completed_or_stopped").wait(
                    TransformJobName=batch_name, WaiterConfig={"Delay": 15, "MaxAttempts": 120}
                )
            current = sm.describe_transform_job(TransformJobName=batch_name)
            if current["TransformJobStatus"] not in {"Completed", "Failed", "Stopped"}:
                raise RuntimeError("Batch compute is still active; preserve model and retry cleanup.")
            batch_job_started = False
        if batch_model_created:
            current = sm.describe_model(ModelName=batch_name)
            if current["PrimaryContainer"].get("ModelPackageName") != PACKAGE_ARN:
                raise RuntimeError("Batch package identity differs; preserve the task model.")
            sm.delete_model(ModelName=batch_name)
            batch_model_created = False
        print("Batch compute stopped; model cleaned. Task S3 evidence retained:", batch_prefix)